## H(curl) - AMG

The `hcurlamg` is an implementation of the amg from [Reitzinger and Schöberl: An algebraic multigrid method for finite element discretizations with edge elements](https://onlinelibrary.wiley.com/doi/abs/10.1002/nla.271?casa_token=SGxs8UGF--IAAAAA:53O8vbFJpEkXyuSu4T2yzP7BKBJecdNoFdEvLqUKT_ZRUMn0U5FM--SqGXRiQu38et4xuMPg6cPUgfUBoQ).

It is based on a surrogate matrix for a weighted $H(\operatorname{curl})$ norm discretized by lowest order Nedelec elements:

$$
\| u \|_{L_2, \sigma}^2 + \| \operatorname{curl} u \|_{L_2, \nu}^2
\approx \sum_E w_E \, \Big(\int_E u_{\tau} \Big)^2 + 
\sum_F w_F \, \Big(\int_F \operatorname{curl}_n u \Big)^2
$$

The smoother is a Hiptmair smoother, where a Gauss-Seidel smoother is combined with another Gauss-Seidel smoother for the potential space.

The key is a coarsening which preserves the de Rham sequence over all levels, such that Hiptmair's smoother is effective also on coarser levels.

<img src="agglomerates-hc.png" alt="Alternative text" width="300" align="center"/>



More recent, robust coarsening strategies are developed in [B. Schwarzenbacher: Robust algebraic solvers for electromagnetics, Master's Thesis](https://repositum.tuwien.at/handle/20.500.12708/1351)


In [ ]:
from ngsolve import *
from ngsolve.la import EigenValues_Preconditioner

with TaskManager():
    mesh = Mesh(unit_cube.GenerateMesh(maxh=0.1))
    for l in range(1): mesh.Refine()   

In [ ]:
fes = HCurl(mesh, order=0)
print ("ndof = ", fes.ndof)
u,v = fes.TnT()

a = BilinearForm(curl(u)*curl(v)*dx + 0.01*u*v*dx)
pre = Preconditioner(a, "hcurlamg")
with TaskManager():
    a.Assemble()
    lam = EigenValues_Preconditioner(a.mat, pre.mat)
    print (list(lam[0:3]), '...', list(lam[-3:-1]))

In [ ]:
f = LinearForm(curl(v)[2]*dx).Assemble()
gfu = GridFunction(fes)
from ngsolve.krylovspace import CGSolver

inv = CGSolver(a.mat, pre.mat, plotrates=True, maxiter=200)

gfu.vec[:] = inv*f.vec